In [ ]:
import os
import re
import time
import pandas as pd
import requests
from dotenv import load_dotenv

# === Load environment and GitHub tokens ===
env_path = "All_Tokens.env"
load_dotenv(env_path)
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 7) if os.getenv(f"GITHUB_TOKEN_{i}")]
if not tokens:
    raise ValueError("❌ No GitHub tokens found.")
token_index = 0

# === CI Patterns ===
ci_patterns = {
    r'\.travis\.yml$': 'Travis_CI',
    r'\.appveyor\.yml$': 'AppVeyor',
    r'appveyor\.yml$': 'AppVeyor',
    r'circle\.yml$': 'Circle_CI',
    r'\.circleci/config\.yml$': 'Circle_CI',
    r'azure-pipelines\.yml$': 'Azure_Pipelines',
    r'\.github/workflows/.*\.(yml|yaml)$': 'GitHub_Actions',
    r'bitbucket-pipelines\.yml$': 'Bitbucket',
    r'\.gitlab-ci\.yml$': 'GitLab',
    r'Jenkinsfile\.yml$': 'Jenkins',
    r'bitrise\.yml$': 'Bitrise',
    r'bamboo\.yml$': 'Bamboo',
    r'codeship-services\.yml$': 'Codeship',
    r'\.gocd\.yaml$': 'GoCD',
    r'\.cirrus\.yml$': 'Cirrus',
    r'wercker\.yaml$': 'Wercker',
    r'semaphore\.yml$': 'Semaphore',
    r'codemagic\.yaml$': 'Nevercode',
}

# === Paths ===
input_csv = r"C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step3_removal_keyword_output.csv"
output_csv = r"C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step4_ci_detection_output.csv"

# === Load input ===
input_df = pd.read_csv(input_csv)
input_df['html_url'] = input_df['html_url'].astype(str).str.strip()
if os.path.exists(output_csv):
    print("🔁 Resuming from previously saved output.")
    input_df = pd.read_csv(output_csv)
else:
    input_df['yml_detected'] = input_df.get('yml_detected', 'none')

# === Filter valid repos to review ===
input_df['Valid_Repo_Step3'] = input_df['Valid_Repo_Step3'].astype(str).str.lower()
rows_to_review = input_df[(input_df['Valid_Repo_Step3'] == 'yes') & (input_df['yml_detected'] == 'none')].index
print(f"🔍 Starting CI file detection using GitHub Search API: {len(rows_to_review)} repos")

# === Token rotation ===
def get_valid_response(url, verbose=True, max_retries=10):
    global token_index
    token_count = len(tokens)
    sleep_times = []
    retries = 0

    while retries < max_retries:
        token_id = token_index % token_count
        token = tokens[token_id]
        headers = {'Authorization': f'token {token}'}
        token_index += 1  # Move to the next token for next round

        try:
            response = requests.get(url, headers=headers, timeout=20)
        except requests.exceptions.RequestException as e:
            if verbose:
                print(f"❌ Request error with token {token_id + 1}: {e}")
            retries += 1
            continue

        status = response.status_code
        remaining = response.headers.get("X-RateLimit-Remaining", "?")
        if verbose:
            print(f"🔁 Token {token_id + 1} | Status: {status} | Remaining: {remaining}")

        # === Success or not found is acceptable ===
        if status == 200 or status == 404:
            return response, token_id + 1

        # === Rate Limit Hit ===
        elif status == 403 and response.headers.get("X-RateLimit-Remaining") == "0":
            reset_time = int(response.headers.get("X-RateLimit-Reset", time.time() + 60))
            wait_seconds = max(reset_time - int(time.time()), 1)
            if verbose:
                print(f"⏳ Token {token_id + 1} rate-limited. Will retry others.")
            sleep_times.append(wait_seconds)
            retries += 1
            continue

        # === Possibly Abuse Detection or Permission Issue ===
        elif status == 403:
            if verbose:
                print(f"🚫 Token {token_id + 1} got 403 - Possibly abuse detection or private repo.")
            retries += 1
            continue

        # === Unexpected Error ===
        else:
            if verbose:
                print(f"⚠️ Token {token_id + 1} unexpected HTTP error: {status}")
            retries += 1
            continue

    # === All Tokens Failed ===
    if sleep_times:
        min_wait = min(sleep_times)
        if verbose:
            print(f"😴 All tokens exhausted. Sleeping for {min_wait} seconds...")
        time.sleep(min_wait)
        return get_valid_response(url, verbose, max_retries=max_retries)

    if verbose:
        print("❌ All tokens failed without rate-limit. Returning None.")
    return None, None

# === Process each repo ===
for count, idx in enumerate(rows_to_review, start=1):
    url = input_df.at[idx, 'html_url']
    owner, repo = url.rstrip('/').split('/')[-2:]
    print(f"🔎 [{count}/{len(rows_to_review)}] Scanning: {owner}/{repo}")

    matches = []

    # Check both yml and yaml extensions
    for ext in ['yml', 'yaml']:
        query = f"repo:{owner}/{repo} extension:{ext} path:.github/workflows"
        search_url = f"https://api.github.com/search/code?q={query}"
        r, _ = get_valid_response(search_url)
        if not r:
            continue
        for item in r.json().get("items", []):
            path = item.get("path", "")
            for pattern, ci_type in ci_patterns.items():
                if re.search(pattern, path, re.IGNORECASE):
                    matches.append((path, ci_type))
                    break

    # Save result
    input_df.at[idx, 'yml_detected'] = 'yes' if matches else 'no'
    input_df.at[idx, 'total_yml_files'] = len(matches)
    input_df.at[idx, 'yml_paths'] = '; '.join(p for p, _ in matches)
    input_df.at[idx, 'ci_types'] = '; '.join(ci for _, ci in matches)

    # Save after each repo
    input_df.to_csv(output_csv, index=False)

print("\n✅ Search API-based CI detection complete.")


🔍 Starting CI file detection using GitHub Search API: 18102 repos
🔎 [1/18102] Scanning: Dawnthorn/nagare
🔁 Token 1 | Status: 200 | Remaining: 9
🔁 Token 2 | Status: 200 | Remaining: 9
🔎 [2/18102] Scanning: jamplus/jamplus
🔁 Token 3 | Status: 200 | Remaining: 9
🔁 Token 4 | Status: 200 | Remaining: 9
🔎 [3/18102] Scanning: bpellin/keepassdroid
🔁 Token 5 | Status: 200 | Remaining: 9
🔁 Token 6 | Status: 200 | Remaining: 9
🔎 [4/18102] Scanning: samuelclay/NewsBlur
🔁 Token 1 | Status: 200 | Remaining: 8
🔁 Token 2 | Status: 200 | Remaining: 8
🔎 [5/18102] Scanning: connectbot/connectbot
🔁 Token 3 | Status: 200 | Remaining: 8
🔁 Token 4 | Status: 200 | Remaining: 8
🔎 [6/18102] Scanning: JakeWharton/SMSMorse
🔁 Token 5 | Status: 200 | Remaining: 8
🔁 Token 6 | Status: 200 | Remaining: 8
🔎 [7/18102] Scanning: JakeWharton/SMSBarrage
🔁 Token 1 | Status: 200 | Remaining: 7
🔁 Token 2 | Status: 200 | Remaining: 7
🔎 [8/18102] Scanning: millenomi/diceshaker
🔁 Token 3 | Status: 200 | Remaining: 7
🔁 Token 4 | 